# VacxiS: query-driven test-dataset generation with Qwen2.5 + vLLM

This notebook runs the **query-driven** dataset path on a GPU host
(Colab / Kaggle). It:

1. Clones the repo and installs `vllm`.
2. Converts the 1200 raw user queries in `scrape_dataset/query.parquet`
   into vaccine-domain questions grounded on `chunks_storage/chunks.parquet`
   using **Qwen2.5-7B-Instruct** via vLLM.
3. Writes `test_dataset.json` and validates it.

> Set your runtime to a GPU (T4 / A10G / P100). vLLM and Qwen2.5-7B need
> CUDA. On Kaggle enable the T4 GPU in the kernel settings.

In [ ]:
# 1. Get the repo.
# Replace <your-repo-url> with your git URL (public, so Colab can clone it).
REPO = "https://github.com/<your-user>/vacxis"
!git clone --depth 1 $REPO vacxis && cd vacxis
!ls

In [ ]:
# 2. Install vLLM (pulls CUDA wheels; can take a few minutes on first run).
!pip install vllm
!pip install pyarrow openai qdrant-client

In [ ]:
# 3. Sanity-check the required inputs are present.
import os
for p in [
    "chunks_storage/chunks.parquet",
    "scrape_dataset/query.parquet",
    "75_vaccine_names_mapping.json",
]:
    print(p, "->", os.path.exists(p))

In [ ]:
# 4. Drive the CLI (vLLM path). Model auto-downloads from HuggingFace, or
#    point --vllm-model at a local weights dir you've mounted.
!python generate_test_dataset.py \
    --source queries \
    --source-queries scrape_dataset/query.parquet \
    --vllm \
    --vllm-model Qwen/Qwen2.5-7B-Instruct \
    --distractors 2 \
    --seed 42 \
    --out test_dataset.json
# (equivalently: uv run generate_test_dataset.py ... once `uv` is available)

In [ ]:
# 5. Inspect the result.
import json
data = json.load(open("test_dataset.json"))
print("records:", len(data))
print(json.dumps(data[0], ensure_ascii=False, indent=2)[:1200])

## No-GPU fallback (CPU only)

If you have no GPU, run the **offline rule-based** path (no vLLM, no model
download) — it keeps each original query and grounds it on a random corpus
vaccine with extractive answers:

```bash
python generate_test_dataset.py --source queries --no-vllm --out test_dataset.json
```

## Legacy template generator

```bash
python generate_test_dataset.py --source templates --per-type 8
```